### <font color='#F5DEB3'>1.1 Подключение библиотек

1. Для выполнения практикума рекомендуется использовать полносвязную нейронную сеть прямого распространения – MLPClassifier из библиотеки машинного обучения sklearn (см. описание алгоритма на странице библиотеки: https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html).
Допускается использовать другие библиотеки машинного обучения, если у Вас есть опыт работы с ними, однако, алгоритм должен остаться непременно тем, который задан в данном задании – полносвязная нейронная сеть прямого распространения.
Далее приведена инструкция из предположения использования библиотеки sklearn для реализации нейросетевого классификатора.

In [1]:
import sys
import os

project_root = os.path.abspath('..')
sys.path.append(project_root)

In [2]:
import sys
import os

project_root = os.path.abspath('..')
sys.path.append(project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import yaml

from config.config import robot

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from sklearn.preprocessing import MinMaxScaler
from imblearn.over_sampling import SMOTE, ADASYN

In [3]:
df = pd.read_excel(robot.DATASET_PATH)
display(df.head(), df.info(), df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 176 entries, 0 to 175
Data columns (total 16 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   I1      176 non-null    float64
 1   I2      176 non-null    float64
 2   I3      176 non-null    float64
 3   gx      176 non-null    float64
 4   gy      176 non-null    float64
 5   gz      176 non-null    float64
 6   ax      176 non-null    float64
 7   ay      176 non-null    float64
 8   az      176 non-null    float64
 9   V1real  176 non-null    float64
 10  V2real  176 non-null    float64
 11  V3real  176 non-null    float64
 12  N1      176 non-null    int64  
 13  N2      176 non-null    int64  
 14  N3      176 non-null    int64  
 15  Type    176 non-null    int64  
dtypes: float64(12), int64(4)
memory usage: 22.1 KB


,I1,I2,I3,gx,gy,gz,ax,ay,az,V1real,V2real,V3real,N1,N2,N3,Type
0,1.126697,0.021116,0.927601,0.712941,-0.252941,1.081765,0.043529,0.003174,0.011661,-249.352941,4.764706,265.235294,-5070,30,5239,1
1,1.039215,0.015083,0.953243,0.229412,0.276471,0.485294,0.013672,-0.005572,0.012164,-260.470588,0.000000,263.647059,-5320,-9,5391,1
2,0.911011,0.004525,0.947210,1.170588,0.222941,0.664118,0.011704,0.003720,0.005744,-268.411765,0.000000,268.411765,-4829,6,4786,1
3,0.692308,1.156862,0.550528,1.165294,0.698235,-3.421765,0.001321,-0.010311,-0.001522,141.352941,-306.529412,162.000000,2670,-5662,2875,1
4,0.529412,1.024133,0.453997,0.808235,0.090000,-1.449412,0.006218,-0.010986,-0.006247,155.647059,-314.470588,169.941176,2841,-5824,2967,1


None

,I1,I2,I3,gx,gy,gz,ax,ay,az,V1real,V2real,V3real,N1,N2,N3,Type
count,176.000000,176.000000,176.000000,176.000000,176.000000,176.000000,176.000000,176.000000,176.000000,176.000000,176.000000,176.000000,176.000000,176.000000,176.000000,176.000000
mean,0.489130,0.496730,0.515435,0.489197,-0.918759,21.789526,0.002805,-0.002898,-0.000539,165.642524,104.380920,341.394548,2816.715909,1649.357955,5811.000000,3.011364
std,0.308620,0.408217,0.299098,1.937254,1.837406,28.148416,0.017431,0.024498,0.022011,314.381930,358.226896,210.213155,5370.894465,6209.505848,3557.591498,1.426238
min,0.037607,0.000000,0.029060,-16.461250,-6.236250,-4.251176,-0.060758,-0.094824,-0.116547,-300.600000,-363.600000,-4.764706,-5519.000000,-8010.000000,-72.000000,1.000000
25%,0.273050,0.241026,0.296795,-0.052768,-1.613667,0.552147,-0.004743,-0.012504,-0.009770,-108.450000,-257.294118,168.023077,-1830.750000,-4892.000000,2914.500000,2.000000
50%,0.383629,0.383760,0.403419,0.488000,-0.189101,6.161208,0.002562,0.000008,0.000168,155.647059,57.375000,300.600000,2732.500000,1022.500000,4895.500000,3.000000
75%,0.696154,0.727800,0.780141,0.873414,0.276618,37.439714,0.012108,0.008070,0.009354,344.250000,339.621429,450.140625,5674.000000,5621.750000,7796.000000,4.000000
max,1.365008,1.561085,1.223229,9.905385,3.088000,87.827333,0.100601,0.100656,0.087982,772.200000,768.600000,786.600000,14770.000000,14829.000000,14554.000000,5.000000


In [6]:
dt = 0.01  # период дискретизации (уточните по вашим данным)
df_alpha = np.cumsum(df['gz'].values) * dt

In [ ]:
import numpy as np
import pandas as pd

def compute_V2_V3(df, df_alpha):
    theta = np.radians(robot.WHEEL_ANGLE)          # угол установки колёс
    alpha = df_alpha                               # массив углов поворота робота

    # Знаки скоростей вращения для коррекции знаков токов
    omega_arr = df[['V1real', 'V2real', 'V3real']].values
    sign_omega = np.sign(omega_arr)

    # ----- Матрица скоростей -----
    c = np.cos(alpha)
    s = np.sin(alpha)
    ct = np.cos(theta)
    st = np.sin(theta)

    a11 = - (2/3) * np.cos(alpha - theta)
    a12 =   (2/3) * np.sin(alpha)             
    a13 =   (2/3) * np.cos(alpha + theta)

    a21 = - (2/3) * np.sin(alpha - theta) 
    a22 = - (2/3) * np.cos(alpha)               
    a23 =   (2/3) * np.sin(alpha + theta)  

    a31 = 1/(3*robot.DIST_CENTER)
    a32 = 1/(3*robot.DIST_CENTER)
    a33 = 1/(3*robot.DIST_CENTER)

    M_vel = np.array([[a11, a12, a13],
                      [a21, a22, a23],
                      [a31, a32, a33]]) 
    M_vel = np.moveaxis(M_vel, 2, 0)          # теперь размер (n,3,3)

    # ----- Матрица для токов -----
    b31 = 1/3
    b32 = 1/3
    b33 = 1/3

    M_curr = np.array([[a11, a12, a13],
                       [a21, a22, a23],
                       [b31, b32, b33]])
    M_curr = np.moveaxis(M_curr, 2, 0)        # (n,3,3)

    # ----- Вычисление Vx, Vy, Ω -----
    V = np.einsum('ijk,ik->ij', M_vel, omega_arr) * robot.WHEEL_RADIUS   # (n,3)

    # ----- Вычисление Ix, Iy, Iφ -----
    I_arr = df[['I1', 'I2', 'I3']].values
    I_signed = I_arr * sign_omega
    I_vec = np.einsum('ijk,ik->ij', M_curr, I_signed)   # (n,3)

    # ----- IΣ (суммарная трудоёмкость) -----
    I_sum = np.sum(np.abs(I_arr), axis=1)

    # ----- Вычисление V3 (относительные параметры) -----
    eps = 1e-9
    Tx = V[:,0] / (I_vec[:,0] + eps)
    Ty = V[:,1] / (I_vec[:,1] + eps)
    Tphi = V[:,2] / (I_vec[:,2] + eps)
    Tz = df['gz'].values / (I_vec[:,2] + eps)

    # Формируем DataFrame V2
    V2 = pd.DataFrame({
        'Vx': V[:,0],
        'Vy': V[:,1],
        'Ω': V[:,2],
        'Ix': I_vec[:,0],
        'Iy': I_vec[:,1],
        'Iφ': I_vec[:,2],
        'IΣ': I_sum
    }, index=df.index)

    # Формируем DataFrame V3
    V3 = pd.DataFrame({
        'Tx': Tx,
        'Ty': Ty,
        'Tφ': Tphi,
        'Tz': Tz
    }, index=df.index)

    return V2, V3

In [12]:
V2, V3 = compute_V2_V3(df, df_alpha)
display(V2, V3)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (3, 3) + inhomogeneous part.